# Blog Writer Swarm

A team of five Claude agents that turns a topic into a finished blog post.

| Agent | Job | Hands off to |
|---|---|---|
| **Admin** | Confirms topic, audience, tone, length | Planner |
| **Planner** | Builds the outline | Researcher |
| **Researcher** | Gathers facts per section (optional live web search) | Writer |
| **Writer** | Drafts the full post | Editor |
| **Editor** | Tightens, fact-checks against research, approves or sends back | Writer or finish |

**How the swarm works:** each agent is a system prompt plus tools. One tool is always a *handoff* (`transfer_to_<agent>`). When an agent calls it, the runner switches control to that agent and passes along the shared workspace (topic, outline, research, draft). No agent sees another agent's private reasoning, only the work product.

## 1. Setup
Install the Anthropic SDK and set your API key (`ANTHROPIC_API_KEY`), or log in with `ant auth login`.

In [ ]:
%pip install -q anthropic

In [ ]:
import json
from dataclasses import dataclass, field
from typing import Callable

import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY or your `ant auth login` profile

MODEL = "claude-opus-5-5"
EFFORT = "medium"          # low | medium | high | xhigh | max
USE_WEB_SEARCH = False     # True lets the Researcher search the live web (billed per search)
MAX_TURNS = 30             # safety cap on total model calls

## 2. Shared workspace
Every agent reads from and writes to this. It's what gets passed along on each handoff.

In [ ]:
@dataclass
class Workspace:
    brief: dict = field(default_factory=dict)
    outline: list = field(default_factory=list)
    research: dict = field(default_factory=dict)
    draft: str = ""
    editor_notes: list = field(default_factory=list)
    final: str = ""

    def summary(self) -> str:
        return json.dumps({
            "brief": self.brief,
            "outline": self.outline,
            "research": self.research,
            "draft": self.draft,
            "editor_notes": self.editor_notes,
        }, indent=2)

## 3. Agents and tools
A tool is a JSON schema plus a Python function. Handoff tools return the name of the next agent.

In [ ]:
@dataclass
class Agent:
    name: str
    instructions: str
    tools: list = field(default_factory=list)                     # tool schemas
    handlers: dict[str, Callable] = field(default_factory=dict)   # name -> fn(ws, **input)
    handoffs: list[str] = field(default_factory=list)
    server_tools: list = field(default_factory=list)


def tool(name, description, properties, required=None):
    return {
        "name": name,
        "description": description,
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": properties,
            "required": required or list(properties),
            "additionalProperties": False,
        },
    }


def handoff_tool(target):
    return tool(
        f"transfer_to_{target.lower()}",
        f"Hand control to the {target} agent. Include a short note on what they should do next.",
        {"note": {"type": "string"}},
    )


# ---- workspace tools ----
def save_brief(ws, topic, audience, tone, word_count):
    ws.brief = {"topic": topic, "audience": audience, "tone": tone, "word_count": word_count}
    return "Brief saved."

def save_outline(ws, title, sections):
    ws.outline = [{"title": title}] + sections
    return f"Outline saved with {len(sections)} sections."

def save_research(ws, section, notes):
    ws.research[section] = notes
    return f"Research saved for '{section}'."

def save_draft(ws, markdown):
    ws.draft = markdown
    return f"Draft saved ({len(markdown.split())} words)."

def add_editor_note(ws, note):
    ws.editor_notes.append(note)
    return "Note recorded."

def publish(ws, markdown):
    ws.final = markdown
    return "PUBLISHED"

In [ ]:
STRING = {"type": "string"}

admin = Agent(
    name="Admin",
    instructions=(
        "You run a small blog team. Turn the user's request into a clear brief: topic, "
        "audience, tone, and target word count. Fill sensible defaults for anything missing "
        "and say what you assumed. Save it with save_brief, then transfer to the Planner."
    ),
    tools=[tool("save_brief", "Save the writing brief.",
                {"topic": STRING, "audience": STRING, "tone": STRING,
                 "word_count": {"type": "integer"}})],
    handlers={"save_brief": save_brief},
    handoffs=["Planner"],
)

planner = Agent(
    name="Planner",
    instructions=(
        "You plan blog posts. Read the brief and write an outline of 3 to 6 sections, each "
        "with one clear point. Save it with save_outline, then transfer to the Researcher."
    ),
    tools=[tool("save_outline", "Save the post title and section outline.",
                {"title": STRING,
                 "sections": {"type": "array", "items": {
                     "type": "object",
                     "properties": {"heading": STRING, "key_point": STRING},
                     "required": ["heading", "key_point"],
                     "additionalProperties": False}}})],
    handlers={"save_outline": save_outline},
    handoffs=["Researcher"],
)

researcher = Agent(
    name="Researcher",
    instructions=(
        "You research blog sections. For each outline section, record concrete facts, "
        "examples, and sources with save_research. Never invent statistics or quotes. "
        "If you are unsure of a fact, label it 'unverified'. When every section has notes, "
        "transfer to the Writer."
    ),
    tools=[tool("save_research", "Save research notes for one section.",
                {"section": STRING, "notes": STRING})],
    handlers={"save_research": save_research},
    handoffs=["Writer"],
    server_tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 8}]
                 if USE_WEB_SEARCH else [],
)

writer = Agent(
    name="Writer",
    instructions=(
        "You write blog posts in Markdown. Follow the brief, outline, and research. Use only "
        "facts from the research notes. If editor notes exist, address every one. Save the "
        "full post with save_draft, then transfer to the Editor."
    ),
    tools=[tool("save_draft", "Save the full Markdown draft.", {"markdown": STRING})],
    handlers={"save_draft": save_draft},
    handoffs=["Editor"],
)

editor = Agent(
    name="Editor",
    instructions=(
        "You edit blog posts. Check the draft against the brief and research: accuracy, "
        "clarity, structure, length. If it needs real changes, record each with "
        "add_editor_note and transfer back to the Writer. If it is ready, make light line "
        "edits yourself and call publish with the final Markdown. Send it back at most twice; "
        "after that, fix it yourself and publish."
    ),
    tools=[tool("add_editor_note", "Record one required change for the Writer.", {"note": STRING}),
           tool("publish", "Publish the final post.", {"markdown": STRING})],
    handlers={"add_editor_note": add_editor_note, "publish": publish},
    handoffs=["Writer"],
)

AGENTS = {a.name: a for a in [admin, planner, researcher, writer, editor]}

## 4. The swarm runner

Each time control passes to an agent, it starts a fresh conversation with the current workspace as context. That keeps every agent focused and every conversation append-only.

Notes on the API calls:
- **Adaptive thinking** is on by default for this model; `effort` controls how hard it thinks.
- **Refusal fallbacks** (`fallbacks="default"`) are enabled: if a safety classifier declines a request, the API reruns it on a fallback model instead of stopping. Remove those two arguments if you don't want that.

In [ ]:
def call_model(agent, messages):
    while True:
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=16000,
            system=agent.instructions,
            tools=agent.tools + [handoff_tool(t) for t in agent.handoffs] + agent.server_tools,
            messages=messages,
            output_config={"effort": EFFORT},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
        )
        if response.stop_reason == "pause_turn":   # long server-side web search: let it continue
            messages.append({"role": "assistant", "content": response.content})
            continue
        return response


def run_swarm(request: str, start: str = "Admin", verbose: bool = True) -> Workspace:
    ws = Workspace()
    current, note, turns = start, request, 0

    while current and turns < MAX_TURNS:
        agent = AGENTS[current]
        if verbose:
            print(f"\n=== {agent.name} ===")
        messages = [{"role": "user", "content":
                     f"Request / handoff note:\n{note}\n\nShared workspace:\n{ws.summary()}"}]
        next_agent, nudged = None, False

        while turns < MAX_TURNS:
            turns += 1
            response = call_model(agent, messages)

            if response.stop_reason == "refusal":
                print(f"{agent.name} declined: {response.stop_details}")
                return ws
            if response.stop_reason == "max_tokens":
                print(f"{agent.name} hit max_tokens; stopping.")
                return ws

            messages.append({"role": "assistant", "content": response.content})
            tool_uses = [b for b in response.content if b.type == "tool_use"]

            for b in response.content:
                if b.type == "text" and verbose and b.text.strip():
                    print(f"{agent.name}: {b.text.strip()[:400]}")

            if not tool_uses:
                if ws.final:
                    return ws
                if nudged:
                    print(f"{agent.name} stopped without handing off.")
                    return ws
                nudged = True
                messages.append({"role": "user", "content":
                                 "Finish your job with the tools, then hand off."})
                continue

            results = []
            for tu in tool_uses:
                if tu.name.startswith("transfer_to_"):
                    next_agent = next(n for n in agent.handoffs
                                      if f"transfer_to_{n.lower()}" == tu.name)
                    note = tu.input.get("note", "")
                    out = f"Transferred to {next_agent}."
                elif tu.name in agent.handlers:
                    try:
                        out = agent.handlers[tu.name](ws, **tu.input)
                    except Exception as e:
                        results.append({"type": "tool_result", "tool_use_id": tu.id,
                                        "content": f"Error: {e}", "is_error": True})
                        continue
                else:
                    out = f"Unknown tool {tu.name}"
                if verbose:
                    print(f"  -> {tu.name}: {out}")
                results.append({"type": "tool_result", "tool_use_id": tu.id, "content": out})

            if ws.final:
                return ws
            if next_agent:
                break
            messages.append({"role": "user", "content": results})

        current = next_agent

    return ws

## 5. Run it

In [ ]:
ws = run_swarm(
    "Write a blog post for small-business owners on why their website isn't "
    "turning visitors into customers, and three fixes they can make this week."
)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(ws.final or ws.draft or "_No draft produced._"))

## 6. Make it yours

- **Change the team:** add an agent (e.g. `SEO`) and put its name in another agent's `handoffs`.
- **Live research:** set `USE_WEB_SEARCH = True` and re-run the agent cell.
- **Cost vs. quality:** drop `EFFORT` to `"low"` for quick drafts, raise to `"high"` for polished pieces.
- **Save the post:** `open("post.md", "w").write(ws.final)`.